# 01 — Auditoría inicial y calidad del dataset

**PPIV: Proyecto Integrador 2026 — Ciencia de Datos e Inteligencia Artificial**

Este notebook realiza la auditoría previa a cualquier transformación. El objetivo es conocer la estructura del dataset, medir su calidad, identificar inconsistencias y dejar documentadas las reglas que deberán resolverse en la etapa de limpieza.

> **Principio de trabajo:** en esta etapa no se corrigen datos. Se detectan, cuantifican y documentan problemas para evitar decisiones arbitrarias.


## Objetivos de esta etapa

- Verificar estructura, granularidad y tipos de datos.
- Cuantificar valores faltantes y duplicados.
- Revisar la variable objetivo `churn` y el desbalance de clases.
- Analizar rangos y cardinalidad de variables.
- Detectar inconsistencias lógicas y temporales.
- Identificar variables que podrían generar *data leakage*.
- Generar un resumen de calidad que sirva como entrada al notebook de limpieza.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

RANDOM_STATE = 42


In [ ]:
# El notebook puede ejecutarse desde /notebooks o desde la raíz del proyecto.
candidates = [
    Path("../data/raw/sales_marketing_customer_raw.csv"),
    Path("data/raw/sales_marketing_customer_raw.csv"),
]

DATA_PATH = next((p for p in candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "No se encontró sales_marketing_customer_raw.csv. "
        "Ubicarlo en data/raw/ antes de ejecutar el notebook."
    )

df = pd.read_csv(DATA_PATH)
print(f"Dataset cargado desde: {DATA_PATH.resolve()}")
display(df.head())


## 1. Dimensiones, granularidad y esquema

In [ ]:
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
print(f"Duplicados completos: {df.duplicated().sum():,}")

if "customer_id" in df.columns:
    print(f"IDs únicos: {df['customer_id'].nunique(dropna=False):,}")
    print(f"IDs duplicados: {df['customer_id'].duplicated().sum():,}")

schema = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "no_nulos": df.notna().sum(),
    "nulos": df.isna().sum(),
    "unicos": df.nunique(dropna=False)
})
display(schema)


**Criterio de granularidad esperado:** una fila por cliente.  
Esto debe verificarse con `customer_id` antes de construir modelos.

## 2. Valores faltantes

In [ ]:
missing = pd.DataFrame({
    "faltantes": df.isna().sum(),
    "porcentaje": (df.isna().mean() * 100).round(2),
}).sort_values(["porcentaje", "faltantes"], ascending=False)

display(missing[missing["faltantes"] > 0])


In [ ]:
missing_plot = missing[missing["faltantes"] > 0].sort_values("porcentaje")
if not missing_plot.empty:
    ax = missing_plot["porcentaje"].plot(kind="barh", figsize=(8, 4), title="% de valores faltantes")
    ax.set_xlabel("%")
    ax.set_ylabel("Variable")
    plt.tight_layout()
    plt.show()


Los faltantes no deben tratarse todos de la misma manera. Por ejemplo, la ausencia de un `coupon_code` puede representar “no utilizó cupón”, mientras que una `age` faltante representa información demográfica desconocida. La estrategia definitiva se documentará en `02_data_cleaning.ipynb`.

## 3. Variable objetivo: churn

In [ ]:
assert "churn" in df.columns, "El dataset debe contener la variable objetivo 'churn'."

churn_counts = df["churn"].value_counts(dropna=False).sort_index()
churn_pct = (df["churn"].value_counts(normalize=True, dropna=False).sort_index() * 100).round(2)
churn_dist = pd.DataFrame({"cantidad": churn_counts, "porcentaje": churn_pct})
display(churn_dist)

if set(df["churn"].dropna().unique()).issubset({0, 1}):
    churn_rate = df["churn"].mean() * 100
    print(f"Tasa de churn: {churn_rate:.2f}%")
else:
    print("ADVERTENCIA: churn contiene valores distintos de 0/1.")


In [ ]:
ax = df["churn"].value_counts().sort_index().plot(
    kind="bar", figsize=(5, 3), title="Distribución de la variable objetivo"
)
ax.set_xlabel("Churn")
ax.set_ylabel("Cantidad de clientes")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


**Implicancia para modelado:** si las clases están desbalanceadas, `accuracy` no será suficiente. Más adelante se evaluarán, como mínimo, **ROC-AUC, PR-AUC, recall, precision, F1, matriz de confusión y calibración de probabilidades**.

## 4. Estadística descriptiva

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()

display(df[numeric_cols].describe().T)

categorical_summary = pd.DataFrame({
    "unicos": df[categorical_cols].nunique(dropna=False),
    "moda": [df[c].mode(dropna=True).iloc[0] if not df[c].mode(dropna=True).empty else np.nan for c in categorical_cols],
    "freq_moda": [df[c].value_counts(dropna=False).iloc[0] if len(df[c]) else 0 for c in categorical_cols]
})
display(categorical_summary.sort_values("unicos", ascending=False))


## 5. Controles de dominio y rangos

In [ ]:
range_checks = []

def add_check(variable, condition, description):
    if variable in df.columns:
        count = int(condition.sum())
        range_checks.append({
            "variable": variable,
            "regla": description,
            "casos_problematicos": count,
            "porcentaje": round(count / len(df) * 100, 2)
        })

if "age" in df.columns:
    add_check("age", df["age"].notna() & ((df["age"] < 18) | (df["age"] > 100)), "Edad fuera de 18-100")

for c in ["email_open_rate", "email_click_rate"]:
    if c in df.columns:
        add_check(c, df[c].notna() & ((df[c] < 0) | (df[c] > 1)), "Tasa fuera de 0-1")

for c in ["total_visits", "avg_session_time", "pages_per_session", "total_spent",
          "avg_order_value", "support_tickets", "delivery_delay_days",
          "marketing_spend_per_user", "lifetime_value", "last_3_month_purchase_freq"]:
    if c in df.columns:
        add_check(c, df[c].notna() & (df[c] < 0), "Valor negativo")

if "satisfaction_score" in df.columns:
    add_check("satisfaction_score",
              df["satisfaction_score"].notna() & ~df["satisfaction_score"].between(1, 5),
              "Satisfacción fuera de 1-5")

if "nps_score" in df.columns:
    add_check("nps_score",
              df["nps_score"].notna() & ~df["nps_score"].between(0, 10),
              "NPS fuera de 0-10")

range_report = pd.DataFrame(range_checks).sort_values("casos_problematicos", ascending=False)
display(range_report)


## 6. Consistencia temporal

In [ ]:
date_cols = ["signup_date", "last_purchase_date"]
date_parse_report = []

for col in date_cols:
    if col in df.columns:
        parsed = pd.to_datetime(df[col], errors="coerce")
        failed = int(parsed.isna().sum() - df[col].isna().sum())
        date_parse_report.append({"variable": col, "fechas_no_parseables": failed})
        df[col] = parsed

display(pd.DataFrame(date_parse_report))

if all(col in df.columns for col in date_cols):
    temporal_errors = df[
        df["signup_date"].notna()
        & df["last_purchase_date"].notna()
        & (df["last_purchase_date"] < df["signup_date"])
    ]
    print(f"Última compra anterior al alta: {len(temporal_errors):,} "
          f"({len(temporal_errors)/len(df)*100:.2f}%)")
    display(temporal_errors[["customer_id", "signup_date", "last_purchase_date"]].head(10))


## 7. Consistencia geográfica

In [ ]:
if {"country", "city"}.issubset(df.columns):
    geo_pairs = (
        df.groupby(["country", "city"], dropna=False)
          .size()
          .reset_index(name="cantidad")
          .sort_values("cantidad", ascending=False)
    )
    print(f"Combinaciones país-ciudad observadas: {len(geo_pairs):,}")
    display(geo_pairs.head(30))


La base no incluye una tabla maestra que permita saber si `country` o `city` es el dato correcto cuando la combinación resulta incoherente. Por lo tanto, **no se deben inventar correcciones geográficas**. En limpieza se evaluará conservar `country` y excluir `city` del modelado si no puede garantizarse su integridad.

## 8. Cardinalidad y categorías

In [ ]:
category_report = []
for col in categorical_cols:
    vc = df[col].value_counts(dropna=False)
    category_report.append({
        "variable": col,
        "n_categorias": df[col].nunique(dropna=False),
        "categoria_mas_frecuente": vc.index[0] if len(vc) else None,
        "frecuencia_max": int(vc.iloc[0]) if len(vc) else 0
    })

display(pd.DataFrame(category_report).sort_values("n_categorias", ascending=False))


## 9. Relación descriptiva inicial con churn

In [ ]:
candidate_numeric = [
    "satisfaction_score", "support_tickets", "total_spent", "avg_order_value",
    "lifetime_value", "total_visits", "last_3_month_purchase_freq",
    "delivery_delay_days", "email_open_rate", "email_click_rate"
]
candidate_numeric = [c for c in candidate_numeric if c in df.columns]

if candidate_numeric:
    churn_means = df.groupby("churn")[candidate_numeric].mean().T
    churn_means.columns = [f"churn_{c}" for c in churn_means.columns]
    if {"churn_0", "churn_1"}.issubset(churn_means.columns):
        churn_means["dif_abs"] = churn_means["churn_1"] - churn_means["churn_0"]
    display(churn_means.sort_values("dif_abs", key=lambda s: s.abs(), ascending=False)
            if "dif_abs" in churn_means.columns else churn_means)


Estas diferencias son **descriptivas**, no prueban causalidad ni significancia estadística. Servirán para formular hipótesis que serán contrastadas posteriormente.

## 10. Riesgo de data leakage

In [ ]:
# Auditoría preventiva: no elimina variables todavía.
# Se marcan variables que deben revisarse antes de entrenar modelos.
leakage_review = pd.DataFrame([
    ["customer_id", "Excluir", "Identificador; no contiene comportamiento generalizable."],
    ["last_purchase_date", "Transformar/Revisar", "Debe convertirse en recencia respecto de una fecha de corte válida."],
    ["lifetime_value", "Revisar", "Verificar que solo use información disponible antes del momento de predicción."],
    ["total_spent", "Revisar", "Verificar ventana temporal y fecha de corte."],
    ["churn", "Target", "Solo puede usarse como variable objetivo; nunca como predictor."]
], columns=["variable", "accion_preliminar", "motivo"])

display(leakage_review[leakage_review["variable"].isin(df.columns)])


## 11. Resumen automático de calidad

In [ ]:
quality_summary = {
    "filas": len(df),
    "columnas": df.shape[1],
    "duplicados_completos": int(df.duplicated().sum()),
    "customer_id_duplicados": int(df["customer_id"].duplicated().sum()) if "customer_id" in df.columns else np.nan,
    "columnas_con_nulos": int((df.isna().sum() > 0).sum()),
    "tasa_churn_pct": round(float(df["churn"].mean() * 100), 2) if "churn" in df.columns else np.nan,
    "errores_temporales": int(len(temporal_errors)) if "temporal_errors" in globals() else np.nan,
    "edades_fuera_18_100": int(
        (df["age"].notna() & ((df["age"] < 18) | (df["age"] > 100))).sum()
    ) if "age" in df.columns else np.nan,
}
display(pd.Series(quality_summary, name="valor").to_frame())


## Decisiones que pasan al notebook de limpieza

1. Preservar una copia inmutable del dataset original.
2. Resolver faltantes según significado de cada variable, no con una regla única.
3. Marcar como faltantes los valores físicamente/lógicamente imposibles cuando corresponda.
4. No “corregir” país-ciudad sin una fuente externa confiable.
5. Resolver las inconsistencias temporales antes de crear variables de recencia y antigüedad.
6. Excluir identificadores del conjunto de predictores.
7. Definir una **fecha de corte** para evitar fuga de información temporal.
8. Construir un pipeline reproducible de preprocessing para los modelos.

**Próximo paso:** `02_data_cleaning.ipynb`.
